# Aggregate Current Store Features - Gold Layer

Aggregates H3 features for existing store trade areas using H3 polyfill.

**Approach:**
1. Load 5-min isochrones for existing stores
2. Polyfill with H3 cells (find all cells whose centers fall inside isochrone)
3. Inner join with h3_features_clean
4. Aggregate by store
5. Join with sales data

**Multi-State Support:** Includes MI, VA, NY, WA, MD, NJ, MA stores for spatial cross-validation.

**Inputs:**
- `{catalog}.{silver_schema}.isochrones_lce` - Store trade area polygons
- `{catalog}.{silver_schema}.h3_features_clean` - Clean H3 features
- `{catalog}.{bronze_schema}.current_stores_raw` - Current stores with sales data

**Output:**
- `{catalog}.{gold_schema}.current_stores_features_agg` - Aggregated features per store with sales data

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, expr, explode, lit
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", "")
dbutils.widgets.text("bronze_schema", "")
dbutils.widgets.text("silver_schema", "")
dbutils.widgets.text("gold_schema", "")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")

# Validate required parameters
assert catalog, "Missing required parameter: catalog"
assert bronze_schema, "Missing required parameter: bronze_schema"
assert silver_schema, "Missing required parameter: silver_schema"
assert gold_schema, "Missing required parameter: gold_schema"

H3_RESOLUTION = 8  # Using H3 resolution 8

# Table names
isochrones_table = f"{catalog}.{silver_schema}.isochrones_lce"
h3_features_table = f"{catalog}.{silver_schema}.h3_features_clean"
current_stores_table = f"{catalog}.{bronze_schema}.current_stores_raw"
output_table = f"{catalog}.{gold_schema}.current_stores_features_agg"

print(f"Input isochrones: {isochrones_table}")
print(f"H3 features: {h3_features_table}")
print(f"Current stores (for sales): {current_stores_table}")
print(f"Output: {output_table}")

## Load Trade Areas

In [ ]:
# Load LCE isochrones
trade_areas = spark.table(isochrones_table)

# Standardize columns
columns = trade_areas.columns
id_col = next((c for c in columns if c in ['store_number', 'location_id', 'id']), columns[0])

base_ta = trade_areas.select(
    col(id_col).alias("store_number"),
    col("latitude"),
    col("longitude"),
    col("store_type"),
    col("city") if "city" in columns else lit(None).alias("city"),
    col("state") if "state" in columns else lit(None).alias("state"),
    col("drive_time_minutes") if "drive_time_minutes" in columns else lit(5).alias("drive_time_minutes"),
    col("area_sqkm") if "area_sqkm" in columns else lit(None).alias("area_sqkm"),
    col("geometry")
)

print(f"Loaded {base_ta.count()} existing LCE store trade areas")
display(base_ta.limit(5))

## H3 Polyfill Trade Areas

Use H3 polyfill to find all cells whose centers fall inside the 5-min isochrone.

In [ ]:
# H3 Polyfill: Find all H3 cells whose centers fall inside each isochrone
# This is cleaner than the hierarchical coarse cover + children + filter approach

ta_h3 = base_ta.withColumn(
    "h3_cell_id",
    explode(expr(f"h3_polyfillash3string(ST_AsBinary(geometry), {H3_RESOLUTION})"))
)

print(f"Trade areas indexed with H3 resolution {H3_RESOLUTION} using polyfill")
print(f"Total H3 cells across all stores: {ta_h3.count():,}")
print(f"\nCells per store:")
display(ta_h3.groupBy("store_number", "state").count().orderBy(F.desc("count")).limit(10))

## Join to H3 Features

In [ ]:
# Load clean H3 features
h3_features = spark.table(h3_features_table)

print(f"Loaded clean H3 features with {h3_features.count():,} cells")
print(f"Available columns: {h3_features.columns}")

# Join trade area H3 cells with clean features
ta_with_features = ta_h3.join(h3_features, "h3_cell_id", "inner")

print(f"\nJoined {ta_with_features.count()} H3 cells with features")
display(ta_with_features.select(
    "store_number", "h3_cell_id", "population", "target_demographic_total", 
    "total_poi_count", "urbanity", "human_activity_index"
).limit(5))

## Aggregate Features by Store

In [ ]:
# Define features to aggregate from h3_features_clean

# POI columns (individual categories) — SUM across trade area
poi_cols = ['retail', 'food_drink', 'leisure', 'education', 'healthcare', 'financial', 'tourism', 'transportation']
existing_poi_cols = [c for c in poi_cols if c in ta_with_features.columns]

# Count-based demographics — SUM across trade area
demographic_sum_cols = ['population', 'target_demographic_total', 'total_households']
existing_demo_sum_cols = [c for c in demographic_sum_cols if c in ta_with_features.columns]

# Rate/median features — AVG across trade area
rate_cols = [
    'human_activity_index', 'median_household_income', 'per_capita_income',
    'median_home_value', 'median_gross_rent',
    'pct_college_educated', 'unemployment_rate', 'total_poi_count_norm'
]
existing_rate_cols = [c for c in rate_cols if c in ta_with_features.columns]

# Count-based activity — SUM
sum_activity_cols = ['total_poi_count']
existing_sum_activity = [c for c in sum_activity_cols if c in ta_with_features.columns]

# Urbanity (categorical - take mode/first value)
urbanity_col = 'urbanity' if 'urbanity' in ta_with_features.columns else None

print(f"POI columns (SUM): {existing_poi_cols}")
print(f"Demographic counts (SUM): {existing_demo_sum_cols}")
print(f"Rate/median features (AVG): {existing_rate_cols}")
print(f"Sum activity: {existing_sum_activity}")
print(f"Urbanity column: {urbanity_col}")

In [ ]:
# Build aggregation expressions
agg_exprs = []

# SUM: count-based demographics (population, households)
for c in existing_demo_sum_cols:
    agg_exprs.append(F.sum(F.coalesce(col(c), lit(0))).cast("long").alias(c))

# SUM: POI counts by category
for c in existing_poi_cols:
    agg_exprs.append(F.sum(F.coalesce(col(c), lit(0))).cast("long").alias(c))

# SUM: total POI count
for c in existing_sum_activity:
    agg_exprs.append(F.sum(F.coalesce(col(c), lit(0))).cast("long").alias(c))

# AVG: rate/median features (income, education %, unemployment, activity index, normalized POI)
for c in existing_rate_cols:
    agg_exprs.append(F.avg(F.coalesce(col(c), lit(0))).alias(c))

# Urbanity - take the mode (most common value in trade area)
if urbanity_col:
    agg_exprs.append(F.first(col(urbanity_col)).alias("urbanity"))

# H3 cell count and geometry
agg_exprs.extend([
    F.count("h3_cell_id").alias("h3_cell_count"),
    F.first("geometry").alias("geometry")
])

# Aggregate by store
ta_features_agg = ta_with_features.groupBy(
    "store_number",
    "latitude",
    "longitude",
    "store_type",
    "city",
    "state",
    "drive_time_minutes",
    "area_sqkm"
).agg(*agg_exprs)

print(f"Aggregated features for {ta_features_agg.count()} stores")
display(ta_features_agg.limit(5))

## Competitor and Partner Density

Count pizza competitors and partner stores within each store's 5-min drive-time trade area.
These features capture local competitive pressure and co-location opportunity.

In [ ]:
# Count pizza competitors and partner stores within each store's trade area
# Uses the H3 polyfill cells from earlier to do efficient spatial joins

competitors_table = f"{catalog}.{silver_schema}.pois_competitors"
partners_table = f"{catalog}.{silver_schema}.pois_partners"

try:
    # Convert competitor/partner POI locations to H3 cells
    competitors_h3 = spark.table(competitors_table).select(
        "poi_id",
        expr("h3_longlatash3string(longitude, latitude, 8)").alias("h3_cell_id")
    )
    partners_h3 = spark.table(partners_table).select(
        "poi_id",
        expr("h3_longlatash3string(longitude, latitude, 8)").alias("h3_cell_id")
    )

    # Count distinct competitor POIs overlapping each store's trade area
    competitor_counts = ta_h3.join(competitors_h3, "h3_cell_id", "inner") \
        .groupBy("store_number").agg(F.countDistinct("poi_id").alias("competitor_count"))

    # Count distinct partner POIs overlapping each store's trade area
    partner_counts = ta_h3.join(partners_h3, "h3_cell_id", "inner") \
        .groupBy("store_number").agg(F.countDistinct("poi_id").alias("partner_count"))

    # Join to aggregated features
    ta_features_agg = ta_features_agg \
        .join(competitor_counts, "store_number", "left") \
        .join(partner_counts, "store_number", "left") \
        .fillna(0, subset=["competitor_count", "partner_count"])

    print(f"Added competitor_count and partner_count features")
    display(ta_features_agg.select("store_number", "state", "competitor_count", "partner_count")
            .orderBy(F.desc("competitor_count")).limit(10))

except Exception as e:
    print(f"Could not add competitor/partner counts (tables may not exist yet): {e}")
    ta_features_agg = ta_features_agg \
        .withColumn("competitor_count", lit(0)) \
        .withColumn("partner_count", lit(0))

## Write to Gold

In [ ]:
# Join annual_sales from current_stores_raw
current_stores = spark.table(current_stores_table).select("location_id", "annual_sales", "monthly_sales")

# Join sales data
ta_features_with_sales = ta_features_agg.join(
    current_stores,
    ta_features_agg["store_number"] == current_stores["location_id"],
    "left"
).drop("location_id")

print(f"Joined sales data for {ta_features_with_sales.count()} stores")

# Add processing timestamp and fill nulls
ta_features_final = ta_features_with_sales.withColumn("processing_timestamp", F.current_timestamp())

numeric_cols = [
    field.name for field in ta_features_final.schema.fields 
    if field.dataType.typeName() in ['long', 'double', 'integer', 'float']
    and field.name not in ['latitude', 'longitude', 'drive_time_minutes', 'area_sqkm', 'annual_sales', 'monthly_sales']
]
ta_features_final = ta_features_final.fillna(0, subset=numeric_cols)

# Add aliased POI columns for backward compatibility
poi_cols = ['retail', 'food_drink', 'leisure', 'education', 'healthcare', 'financial', 'tourism', 'transportation']
for poi_col in poi_cols:
    if poi_col in ta_features_final.columns:
        aliased_name = f"total_{poi_col}_pois"
        ta_features_final = ta_features_final.withColumn(aliased_name, col(poi_col))

print(f"Added aliased POI columns for backward compatibility")

# Deduplicate
window_spec = Window.partitionBy("store_number").orderBy(F.desc("processing_timestamp"))
ta_features_final = ta_features_final.withColumn(
    "row_num", F.row_number().over(window_spec)
).filter(F.col("row_num") == 1).drop("row_num")

print(f"Records to write: {ta_features_final.count()}")
print(f"Final columns: {ta_features_final.columns}")

# Display sample before writing
print("\nSample of final aggregated features (5 rows):")
display(ta_features_final.limit(5))

# Write to gold
(
    ta_features_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(output_table)
)

print(f"✓ Written to {output_table}")

## Summary Statistics

In [ ]:
print("Current Store Baseline Metrics (with Sales Data):")
display(spark.sql(f"""
  SELECT
    COUNT(*) as total_stores,
    ROUND(AVG(area_sqkm), 2) as avg_trade_area_sqkm,
    ROUND(AVG(population), 0) as avg_population,
    ROUND(AVG(target_demographic_total), 0) as avg_target_demo,
    ROUND(AVG(total_households), 0) as avg_households,
    ROUND(AVG(median_household_income), 0) as avg_median_hh_income,
    ROUND(AVG(per_capita_income), 0) as avg_per_capita_income,
    ROUND(AVG(median_home_value), 0) as avg_median_home_value,
    ROUND(AVG(pct_college_educated), 3) as avg_pct_college,
    ROUND(AVG(unemployment_rate), 3) as avg_unemployment,
    ROUND(AVG(total_poi_count), 0) as avg_total_poi,
    ROUND(AVG(human_activity_index), 2) as avg_activity_index,
    ROUND(AVG(annual_sales), 0) as avg_annual_sales,
    ROUND(MIN(annual_sales), 0) as min_annual_sales,
    ROUND(MAX(annual_sales), 0) as max_annual_sales,
    ROUND(AVG(h3_cell_count), 0) as avg_h3_cells
  FROM {output_table}
"""))

print("\nBy State (for Spatial Cross-Validation):")
display(spark.sql(f"""
  SELECT
    state,
    COUNT(*) as store_count,
    ROUND(AVG(population), 0) as avg_pop,
    ROUND(AVG(median_household_income), 0) as avg_hh_income,
    ROUND(AVG(pct_college_educated), 3) as avg_pct_college,
    ROUND(AVG(annual_sales), 0) as avg_sales
  FROM {output_table}
  GROUP BY state
  ORDER BY state
"""))

print("\nBy Urbanity:")
display(spark.sql(f"""
  SELECT
    urbanity,
    COUNT(*) as store_count,
    ROUND(AVG(population), 0) as avg_pop,
    ROUND(AVG(median_household_income), 0) as avg_hh_income,
    ROUND(AVG(annual_sales), 0) as avg_sales
  FROM {output_table}
  GROUP BY urbanity
  ORDER BY avg_sales DESC
"""))